In [1]:
import pandas as pd
import numpy as np


# ==========================================
# 1. 基本設定
# ==========================================

# 乱数のシード値（毎回同じデータを生成するため）
np.random.seed(42)

# データの期間と間隔
start = "2026-01-01 00:00:00"
periods = 7 * 24 * 60

# 1分間隔の日時データ
datetimes = pd.date_range(
    start=start,
    periods=periods,
    freq="min"
)


# ==========================================
# 2. 製造ラインごとの基本設定
# ==========================================

line_settings = {
    "A": {
        "temperature": (75, 2.5),
        "pressure": (300, 5),
        "vibration": (2.0, 0.2)
    },
    "B": {
        "temperature": (65, 1.0),
        "pressure": (290, 5),
        "vibration": (2.2, 0.25)
    },
    "C": {
        "temperature": (70, 2.0),
        "pressure": (295, 5),
        "vibration": (2.5, 0.8)
    }
}


# ==========================================
# 3. データ生成
# ==========================================

data_list = []

# ラインごとにデータを生成
for line, settings in line_settings.items():

    # 時間経過を表す配列（0～7日）
    time_index = np.arange(periods)

    # 1日の周期的な変化
    daily_cycle = np.sin(
        2 * np.pi * time_index / (24 * 60)
    )

    # --------------------------------------
    # 温度データ
    # --------------------------------------

    temp_mean, temp_std = settings["temperature"]

    temperature = (
        temp_mean
        + daily_cycle * 1.0
        + np.random.normal(0, temp_std, periods)
    )

    # ラインA：特定時間帯に温度が急上昇
    if line == "A":
        mask = (
            (datetimes >= "2026-01-03 14:00") &
            (datetimes < "2026-01-03 14:10")
        )
        temperature[mask] += 15

    # --------------------------------------
    # 圧力データ
    # --------------------------------------

    pressure_mean, pressure_std = settings["pressure"]

    pressure = (
        pressure_mean
        + np.random.normal(0, pressure_std, periods)
    )

    # --------------------------------------
    # 振動データ
    # --------------------------------------

    vibration_mean, vibration_std = settings["vibration"]

    vibration = (
        vibration_mean
        + np.random.normal(0, vibration_std, periods)
    )

    # --------------------------------------
    # ラインC：圧力と振動が同時に急上昇
    # --------------------------------------

    if line == "C":
        mask = (
            (datetimes >= "2026-01-05 10:00") &
            (datetimes < "2026-01-05 10:05")
        )

        pressure[mask] += 50
        vibration[mask] += 4

    # --------------------------------------
    # DataFrameに格納
    # --------------------------------------

    df_line = pd.DataFrame({
        "datetime": datetimes,
        "line": line,
        "temperature": temperature,
        "pressure": pressure,
        "vibration": vibration
    })

    data_list.append(df_line)


# ==========================================
# 4. 全ラインのデータを結合
# ==========================================

df = pd.concat(data_list, ignore_index=True)

# 日時・ラインの順番で並び替え
df = df.sort_values(
    ["datetime", "line"]
).reset_index(drop=True)


# ==========================================
# 5. CSV出力
# ==========================================

df.to_csv(
    "production_data.csv",
    index=False
)

print("CSVファイルを出力しました。")
print(f"レコード数：{len(df):,}")
print(df.head())


CSVファイルを出力しました。
レコード数：30,240
             datetime line  temperature    pressure  vibration
0 2026-01-01 00:00:00    A    76.241785  309.243315   2.069229
1 2026-01-01 00:00:00    B    63.507404  287.163106   2.043265
2 2026-01-01 00:00:00    C    71.315662  289.474878   1.739436
3 2026-01-01 00:01:00    A    74.658703  293.461787   1.844691
4 2026-01-01 00:01:00    B    64.864993  288.188077   2.157173
